# Часть 1 — свой MCP-сервер для базы данных: сервер и клиент

В части 0 мы собрали игрушечный сервер. Теперь — настоящая задача: дать агенту доступ к **базе данных
банка** (Postgres, ~1 млн транзакций, 1993–1998, суммы в чешских кронах).

План — по шагам:

1. пишем сервер: 3 инструмента поверх базы;
2. подключаемся к нему клиентом и вызываем инструменты руками;
3. проверяем, что сломать базу нельзя;
4. запускаем тот же сервер по сети (HTTP) и подключаемся по адресу.

**Перед началом** (один раз): база в докере и данные в ней —

```bash
docker compose up -d postgres
uv run python scripts/load_berka.py
```

## Шаг 1. Сервер

Три инструмента — ровно то, что нужно аналитику, который видит базу впервые:

| инструмент | зачем |
|---|---|
| `list_tables()` | какие таблицы есть и что в них |
| `describe_table(table)` | какие колонки в таблице и что они значат |
| `run_sql(sql)` | выполнить запрос и получить ответ |

Ячейка ниже **записывает сервер в файл** `db_server.py` (магия `%%writefile`), чтобы его можно было
запустить отдельной программой. Обратите внимание на `BANK_DSN`: сервер ходит в базу ролью
`mcp_reader`, которой Postgres разрешает **только читать**.

In [1]:
%%writefile db_server.py
"""A minimal MCP server over the bank database (written by 01_mcp_server_client.ipynb)."""

import os
import sys

import psycopg
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()
DSN = os.getenv("BANK_DSN")              # the read-only role mcp_reader
mcp = FastMCP("bank-lite", log_level="WARNING")


def query(sql: str, params=None) -> str:
    """Run SQL and return the result as text: a header line, then one line per row (max 50)."""
    with psycopg.connect(DSN) as conn:
        cursor = conn.execute(sql, params)
        lines = [" | ".join(column.name for column in cursor.description)]
        lines += [" | ".join(str(value) for value in row) for row in cursor.fetchmany(50)]
    return "\n".join(lines)


@mcp.tool()
def list_tables() -> str:
    """List the tables of the bank database with a short description of each."""
    return query("SELECT relname AS table_name, obj_description(oid) AS description "
                 "FROM pg_class WHERE relnamespace = 'public'::regnamespace AND relkind = 'r' ORDER BY 1")


@mcp.tool()
def describe_table(table: str) -> str:
    """Show the columns of one table: name, type and what the column means."""
    return query("SELECT column_name, data_type, col_description(%s::regclass, ordinal_position) AS meaning "
                 "FROM information_schema.columns WHERE table_name = %s ORDER BY ordinal_position",
                 (table, table))


@mcp.tool()
def run_sql(sql: str) -> str:
    """Run a read-only SQL query (PostgreSQL) and return up to 50 rows."""
    try:
        return query(sql)
    except psycopg.Error as error:
        return f"SQL error: {error}"     # the model reads the error and fixes its query


if __name__ == "__main__":
    if "--http" in sys.argv:
        mcp.settings.port = 8011
        mcp.run(transport="streamable-http")   # a network service: http://localhost:8011/mcp
    else:
        mcp.run()                              # stdio: the client starts us as a subprocess

Overwriting db_server.py


## Шаг 2. Клиент

Клиент сам запускает сервер как процесс (`python db_server.py`) и общается с ним через stdin/stdout —
транспорт **stdio**, как в части 0. Порядок всегда один:

1. `stdio_client(...)` — запустить сервер и подключиться к нему;
2. `ClientSession(...)` + `initialize()` — «рукопожатие»;
3. `list_tools()` / `call_tool(...)` — работать.

Когда блок `async with` заканчивается, сервер останавливается.

In [2]:
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command=sys.executable, args=["db_server.py"])

async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools = await session.list_tools()
        for tool in tools.tools:
            print(f"{tool.name:<15} {tool.description}")

list_tables     List the tables of the bank database with a short description of each.
describe_table  Show the columns of one table: name, type and what the column means.
run_sql         Run a read-only SQL query (PostgreSQL) and return up to 50 rows.


## Шаг 3. Вызываем инструменты руками

Делаем то же, что потом будет делать агент: смотрим таблицы → колонки → пишем SQL.

In [3]:
async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        result = await session.call_tool("list_tables", {})
        print(result.content[0].text, "\n")

        result = await session.call_tool("describe_table", {"table": "loans"})
        print(result.content[0].text, "\n")

        result = await session.call_tool("run_sql", {"sql": "SELECT status, count(*) FROM loans GROUP BY status"})
        print(result.content[0].text)

table_name | description
accounts | Bank accounts. Balance is NOT stored here — take the balance of the latest transaction.
cards | Payment cards issued to an account disposition.
clients | Bank clients (people). Linked to accounts through dispositions.
dispositions | Who may use which account (client ↔ account link).
districts | Czech districts where clients live and accounts are opened (demographics).
loans | Loans granted to accounts. Amounts in CZK.
orders | Standing payment orders (regular automatic payments from an account). CZK.
transactions | All account transactions 1993–1998 (~1M rows). Amounts are positive; direction says in/out. CZK. 

column_name | data_type | meaning
loan_id | integer | Loan id.
account_id | integer | → accounts.
granted_on | date | Date the loan was granted.
amount | numeric | Loan amount, CZK.
duration_months | integer | Loan duration in months.
monthly_payment | numeric | Monthly installment, CZK.
status | text | finished_ok | finished_default (not pai

Ответ инструмента — `result.content`: список блоков, у нас один текстовый. Клиент не знает ничего
про Postgres, пароль или SQL-драйвер — всё это живёт **на сервере**. Клиент знает только имена
инструментов и их аргументы.

## Шаг 4. А если попросить удалить данные?

Модель может ошибиться, а пользователь — попросить лишнего. Проверим:

In [4]:
async with stdio_client(server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        for sql in ["DELETE FROM loans", "SELECT nope FROM loans"]:
            result = await session.call_tool("run_sql", {"sql": sql})
            print(f"{sql:<25} → {result.content[0].text.splitlines()[0]}")

DELETE FROM loans         → SQL error: cannot execute DELETE in a read-only transaction
SELECT nope FROM loans    → SQL error: column "nope" does not exist


- `DELETE` не прошёл: роль `mcp_reader` в Postgres **не может писать** — это и есть защита.
  Не «попросить модель не удалять», а не дать ей такой возможности;
- ошибка в SQL возвращается как обычный текст — модель прочитает её и исправит запрос.

> В сервисе из части 5 используется доработанная версия этого сервера — `bank_mcp/server.py`:
> там ещё таймаут запроса, лимит строк и проверка «ровно один SELECT». Идея та же.

## Шаг 5. Тот же сервер — по сети (HTTP)

stdio подходит, когда клиент и сервер на одной машине и клиент сам запускает сервер. Если сервер —
**общий сервис** (в докере, на другой машине, для многих клиентов), его запускают заранее, и он слушает
порт. Наш сервер это уже умеет — флаг `--http`:

In [5]:
import subprocess
import time

http_server = subprocess.Popen([sys.executable, "db_server.py", "--http"])   # works in the background
time.sleep(3)                                                                 # give it a moment to start
print("сервер слушает http://localhost:8011/mcp")

сервер слушает http://localhost:8011/mcp


Клиент теперь не запускает процесс, а подключается **по адресу**. Остальной код — тот же:

In [6]:
from mcp.client.streamable_http import streamable_http_client

async with streamable_http_client("http://localhost:8011/mcp") as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()

        result = await session.call_tool("run_sql", {"sql": "SELECT count(*) AS clients FROM clients"})
        print(result.content[0].text)

clients
5369


In [7]:
http_server.terminate()   # stop the background server

## Итоги

| | stdio | HTTP |
|---|---|---|
| кто запускает сервер | клиент, как подпроцесс | сервер запущен заранее |
| как подключиться | `stdio_client(команда)` | `streamable_http_client(адрес)` |
| для чего | локальные инструменты | общий сервис, докер, много клиентов |

- сервер = функции с `@mcp.tool()`; доступ к базе и пароли — только на сервере;
- клиент: подключиться → `initialize()` → `list_tools()` / `call_tool()`;
- безопасность — правами в базе, а не просьбами к модели.

Дальше — часть 2: вместо нас инструменты будет вызывать LLM.